In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/multiclassificationtask/sample_submission.csv
/kaggle/input/competitions/multiclassificationtask/train.csv
/kaggle/input/competitions/multiclassificationtask/test.csv


In [19]:
# 1. Ma'lumotlarni aniq manzillar orqali o'qib olamiz
train_df = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/multiclassificationtask/test.csv')

# 2. Train to'plamida nechta qator va ustun borligini ko'ramiz
print("Train to'plami o'lchami:", train_df.shape)

# 3. Ustunlar ro'yxatini va ma'lumot turlarini tekshiramiz
train_df.info()

Train to'plami o'lchami: (15000, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             15000 non-null  int64  
 1   N_Days         15000 non-null  float64
 2   Drug           8494 non-null   object 
 3   Age            15000 non-null  float64
 4   Sex            15000 non-null  object 
 5   Ascites        8502 non-null   object 
 6   Hepatomegaly   8492 non-null   object 
 7   Spiders        8491 non-null   object 
 8   Edema          15000 non-null  object 
 9   Bilirubin      15000 non-null  float64
 10  Cholesterol    6701 non-null   float64
 11  Albumin        15000 non-null  float64
 12  Copper         8399 non-null   float64
 13  Alk_Phos       8488 non-null   float64
 14  SGOT           8486 non-null   float64
 15  Tryglicerides  6666 non-null   float64
 16  Platelets      14436 non-null  float64
 17  Prothrombin  

In [20]:
train_df.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,2178.0,D-penicillamine,16374.0,F,N,N,N,N,0.5,263.0,3.20,43.0,1110.0,106.95,67.0,430.0,9.6,3.0,C
1,1,2644.0,D-penicillamine,17774.0,F,N,N,N,N,0.8,280.0,3.60,22.0,678.0,62.00,80.0,427.0,13.0,3.0,C
2,2,3069.0,Placebo,17844.0,F,N,N,N,N,1.1,408.0,4.40,54.0,2108.0,142.60,137.0,203.0,10.6,3.0,C
3,3,2216.0,Placebo,19221.0,F,N,Y,Y,N,0.8,252.0,3.70,36.0,843.0,55.80,56.0,336.0,9.6,4.0,C
4,4,2256.0,Placebo,21600.0,F,N,N,N,N,4.7,348.0,3.06,464.0,961.0,120.90,146.0,298.0,11.0,2.0,D


A) Umumiy va Demografik ma'lumotlar:

id: Bemorning tartib raqami (buni modelga bermaymiz, foydasi yo'q).

N_Days: Bemor ro'yxatga olingandan boshlab, uning holati oxirgi marta tekshirilguncha o'tgan kunlar soni.

Age: Bemorning yoshi (kun hisobida berilgan, buni keyinchalik yilga o'girsak ham bo'ladi).

Sex: Bemorning jinsi (M - erkak, F - ayol).

B) Kasallik belgilari (Simptomlar):

Ascites: Qorin bo'shlig'iga suyuqlik yig'ilishi (Y - bor, N - yo'q).

Hepatomegaly: Jigarning kattalashishi (Y - bor, N - yo'q).

Spiders: Terida qon tomir yulduzchalari paydo bo'lishi (Y - bor, N - yo'q).

Edema: Shishlar borligi (N - shish yo'q, Y - shish bor, S - shish bor lekin dorilar yordamida kamaytirilgan).

D) Tibbiy tahlillar (Laboratoriya natijalari):

Drug: Bemor qabul qilgan dori turi (D-penicillamine yoki Placebo - shunchaki vitamin).

Bilirubin: Qondagi bilirubin miqdori (jigar ishdan chiqqanda bu ko'rsatkich ko'tariladi).

Cholesterol: Qondagi xolesterin miqdori.

Albumin: Qondagi albumin oqsili (jigar yaxshi ishlasa, albumin yuqori bo'ladi).

Copper: Siydikdagi mis miqdori.

Alk_Phos: Ishqoriy fosfataza fermenti miqdori.

SGOT: Jigar fermenti (AST ko'rsatkichi).

Tryglicerides: Qondagi triglitseridlar (yog'lar) miqdori.

Platelets: Trombotsitlar soni (qon ivishi uchun javobgar hujayralar).

Prothrombin: Qonning ivish vaqti (sekundda).

Stage: Kasallikning bosqichi (1, 2, 3 yoki 4-bosqich. 4 — eng og'ir bosqich).

In [21]:
# 1. Kategoriyali ustunlardagi bo'shliqlarni eng ko'p takrorlangan qiymat (mode) bilan to'ldiramiz
cat_cols_with_nan = ['Drug', 'Ascites', 'Hepatomegaly', 'Spiders']
for col in cat_cols_with_nan:
    mode_value = train_df[col].mode()[0]
    train_df[col] = train_df[col].fillna(mode_value)
    test_df[col] = test_df[col].fillna(mode_value)

In [22]:
# 2. Tibbiy tahlillarni kasallik bosqichiga ('Stage') qarab guruhlagan holda median ko'rsatkich bilan to'ldiramiz
# Bu usul har bir bosqichdagi bemorning real holatiga yaqinroq raqam beradi
num_cols_with_nan = ['Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides']
for col in num_cols_with_nan:
    # Train uchun Stage bo'yicha median hisoblaymiz
    train_df[col] = train_df.groupby('Stage')[col].transform(lambda x: x.fillna(x.median()))
    # Test uchun ham xuddi shu mantiqni qo'llaymiz
    test_df[col] = test_df.groupby('Stage')[col].transform(lambda x: x.fillna(x.median()))

In [23]:
# 3. Qolgan juda kam bo'sh joyi bor ustunlarni umumiy median bilan to'ldiramiz
remaining_cols = ['Platelets', 'Prothrombin']
for col in remaining_cols:
    median_value = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)

# 4. Tekshirib ko'ramiz: Bo'sh joylar qoldimi?
print("Train'da qolgan bo'shliqlar soni:", train_df.isnull().sum().sum())
print("Test'da qolgan bo'shliqlar soni:", test_df.isnull().sum().sum())

Train'da qolgan bo'shliqlar soni: 0
Test'da qolgan bo'shliqlar soni: 0


In [24]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             15000 non-null  int64  
 1   N_Days         15000 non-null  float64
 2   Drug           15000 non-null  object 
 3   Age            15000 non-null  float64
 4   Sex            15000 non-null  object 
 5   Ascites        15000 non-null  object 
 6   Hepatomegaly   15000 non-null  object 
 7   Spiders        15000 non-null  object 
 8   Edema          15000 non-null  object 
 9   Bilirubin      15000 non-null  float64
 10  Cholesterol    15000 non-null  float64
 11  Albumin        15000 non-null  float64
 12  Copper         15000 non-null  float64
 13  Alk_Phos       15000 non-null  float64
 14  SGOT           15000 non-null  float64
 15  Tryglicerides  15000 non-null  float64
 16  Platelets      15000 non-null  float64
 17  Prothrombin    15000 non-null  float64
 18  Stage 

In [25]:
# Harfli (object) ustunlar ro'yxati
categorical_columns = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Status']

print("="*50)
print("HAR BIR USTUNDAGI TAKRORLANMAS QIYMATLAR VA ULARNING SONI:")
print("="*50)

for col in categorical_columns:
    print(f"\n--- {col} ustuni ---")
    print(train_df[col].value_counts())

HAR BIR USTUNDAGI TAKRORLANMAS QIYMATLAR VA ULARNING SONI:

--- Drug ustuni ---
Drug
D-penicillamine    11116
Placebo             3884
Name: count, dtype: int64

--- Sex ustuni ---
Sex
F    14392
M      608
Name: count, dtype: int64

--- Ascites ustuni ---
Ascites
N    14614
Y      386
Name: count, dtype: int64

--- Hepatomegaly ustuni ---
Hepatomegaly
N    11142
Y     3858
Name: count, dtype: int64

--- Spiders ustuni ---
Spiders
N    13278
Y     1722
Name: count, dtype: int64

--- Edema ustuni ---
Edema
N    13845
S      792
Y      363
Name: count, dtype: int64

--- Status ustuni ---
Status
C     10053
D      4565
CL      381
Y         1
Name: count, dtype: int64


In [26]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# 1. Status ustunidagi o'sha adashib qolgan 1 dona 'Y' qiymatini 'C' ga almashtiramiz
train_df['Status'] = train_df['Status'].replace({'Y': 'C'})

# 2. 2 ta qiymati bor ustunlarni One-Hot Encoding qilamiz (drop_first=True orqali)
# Bu usul ustunlar soni ko'payib ketishining oldini oladi va model uchun eng qulay format hisoblanadi
binary_cols = ['Drug', 'Sex', 'Ascites', 'Hepatomegaly', 'Spiders']

train_df = pd.get_dummies(train_df, columns=binary_cols, drop_first=True, dtype=int)
test_df = pd.get_dummies(test_df, columns=binary_cols, drop_first=True, dtype=int)

# 3. 2 tadan ko'p qiymati bor 'Edema' ustunini LabelEncoder orqali raqamlaymiz
le_edema = LabelEncoder()
train_df['Edema'] = le_edema.fit_transform(train_df['Edema'])
test_df['Edema'] = le_edema.transform(test_df['Edema'])

# 4. Target (Status) ustunini LabelEncoder orqali raqamlaymiz
le_status = LabelEncoder()
train_df['Status'] = le_status.fit_transform(train_df['Status'])

# 5. Natijani tekshirish uchun ustunlar turlarini ko'ramiz
train_df.dtypes

id                  int64
N_Days            float64
Age               float64
Edema               int64
Bilirubin         float64
Cholesterol       float64
Albumin           float64
Copper            float64
Alk_Phos          float64
SGOT              float64
Tryglicerides     float64
Platelets         float64
Prothrombin       float64
Stage             float64
Status              int64
Drug_Placebo        int64
Sex_M               int64
Ascites_Y           int64
Hepatomegaly_Y      int64
Spiders_Y           int64
dtype: object

In [27]:
# 1. Train to'plamidan kiruvchi xususiyatlar (X) va chiquvchi natija (y) ni ajratamiz
X = train_df.drop(columns=['id', 'Status'])
y = train_df['Status']

# 2. Test to'plamidan ham faqat kiruvchi xususiyatlarni ajratamiz (id shart emas)
X_test = test_df.drop(columns=['id'])

# 3. Ustunlar bir xilligini va mosligini tekshiramiz
print("Train xususiyatlari o'lchami:", X.shape)
print("Test xususiyatlari o'lchami:", X_test.shape)

Train xususiyatlari o'lchami: (15000, 18)
Test xususiyatlari o'lchami: (10000, 18)


In [28]:
from sklearn.model_selection import train_test_split

# Ma'lumotlarni 80% o'qitish (train) va 20% tekshirish (validation) to'plamlariga ajratamiz
# random_state=42 orqali natija har safar bir xil chiqishini ta'minlaymiz
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("O'qitish to'plami o'lchami (X_train):", X_train.shape)
print("Tekshirish to'plami o'lchami (X_val):", X_val.shape)

O'qitish to'plami o'lchami (X_train): (12000, 18)
Tekshirish to'plami o'lchami (X_val): (3000, 18)


In [29]:
from xgboost import XGBClassifier
from sklearn.metrics import log_loss

# 1. XGBoost modelini logloss mezoniga moslab sozlaymiz
xgb_model = XGBClassifier(
    n_estimators=500,          # Daraxtlar soni
    learning_rate=0.05,        # O'rganish tezligi (overfitting bo'lmasligi uchun kichikroq oldik)
    max_depth=5,               # Daraxtning chuqurligi
    objective='multi:softprob',# Bizga ehtimolliklar (probabilities) kerakligi uchun
    random_state=42,
    eval_metric='mlogloss'     # Musobaqa baholash tizimi (multi-class logloss)
)

# 2. Modelni o'qitamiz
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], # Har bir qadamda validation to'plamida tekshirib boradi
    verbose=100                # Har 100 ta qadamda natijani ekranga chiqaradi
)

# 3. Validation to'plami uchun ehtimolliklarni bashorat qilamiz
xgb_val_preds = xgb_model.predict_proba(X_val)

# 4. Bizning real logloss qiymatimizni hisoblaymiz
xgb_score = log_loss(y_val, xgb_val_preds)
print(f"\n🔥 XGBoost modelining Validation LogLoss natijasi: {xgb_score:.5f}")

[0]	validation_0-mlogloss:0.69737
[100]	validation_0-mlogloss:0.38904
[200]	validation_0-mlogloss:0.37796
[300]	validation_0-mlogloss:0.37512
[400]	validation_0-mlogloss:0.37522
[499]	validation_0-mlogloss:0.37650

🔥 XGBoost modelining Validation LogLoss natijasi: 0.37650


In [30]:
from lightgbm import LGBMClassifier
from sklearn.metrics import log_loss

# 1. LightGBM modelini sozlaymiz
# overfitting'ning oldini olish uchun ba'zi parametrlarni (reg_alpha, reg_lambda) qo'shdik
lgb_model = LGBMClassifier(
    n_estimators=500,          # Daraxtlar soni
    learning_rate=0.05,        # O'rganish tezligi
    max_depth=5,               # Daraxt chuqurligi
    objective='multiclass',    # Ko'p sinfli klassifikatsiya
    random_state=42,
    verbosity=-1,              # Ortiqcha keraksiz loglarni yashirish uchun
    reg_alpha=0.1,             # L1 regulizatsiya (aniqlikni oshirishga yordam beradi)
    reg_lambda=0.1             # L2 regulizatsiya
)

# 2. Modelni o'qitamiz
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)], # Validation to'plamida tekshirish
    callbacks=[]               # LightGBM standart o'qitish jarayoni uchun
)

# 3. Validation to'plami uchun ehtimolliklarni bashorat qilamiz
lgb_val_preds = lgb_model.predict_proba(X_val)

# 4. LightGBM uchun LogLoss qiymatini hisoblaymiz
lgb_score = log_loss(y_val, lgb_val_preds)
print(f"\n🔥 LightGBM modelining Validation LogLoss natijasi: {lgb_score:.5f}")


🔥 LightGBM modelining Validation LogLoss natijasi: 0.38423


In [31]:
from sklearn.metrics import accuracy_score

# 1. XGBoost modeli uchun klasslarni bashorat qilamiz va aniqligini foizda hisoblaymiz
xgb_val_classes = xgb_model.predict(X_val)
xgb_accuracy = accuracy_score(y_val, xgb_val_classes) * 100

# 2. LightGBM modeli uchun klasslarni bashorat qilamiz va aniqligini foizda hisoblaymiz
lgb_val_classes = lgb_model.predict(X_val)
lgb_accuracy = accuracy_score(y_val, lgb_val_classes) * 100

print("="*50)
print("MODELLARNING ANIQLIK DARAJA KO'RSATKICHLARI (%):")
print("="*50)
print(f"🚀 XGBoost modeli Accuracy:  {xgb_accuracy:.2f}%")
print(f"⚡ LightGBM modeli Accuracy: {lgb_accuracy:.2f}%")
print("="*50)

MODELLARNING ANIQLIK DARAJA KO'RSATKICHLARI (%):
🚀 XGBoost modeli Accuracy:  85.13%
⚡ LightGBM modeli Accuracy: 84.80%


In [32]:
from catboost import CatBoostClassifier
from sklearn.metrics import log_loss, accuracy_score

# 1. CatBoost modelini sozlaymiz
cb_model = CatBoostClassifier(
    iterations=600,            # Daraxtlar soni
    learning_rate=0.05,        # O'rganish tezligi
    depth=5,                   # Daraxt chuqurligi
    loss_function='MultiClass',# Ko'p sinfli klassifikatsiya
    random_seed=42,
    verbose=100                # Har 100 ta qadamda natijani ko'rsatadi
)

# 2. Modelni o'qitamiz
cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),   # Validation to'plamida tekshirish
)

# 3. Validation to'plami uchun ehtimollik va klasslarni bashorat qilamiz
cb_val_preds = cb_model.predict_proba(X_val)
cb_val_classes = cb_model.predict(X_val).flatten() # O'lchamini to'g'rilash uchun

# 4. LogLoss va Accuracy ko'rsatkichlarini hisoblaymiz
cb_loss = log_loss(y_val, cb_val_preds)
cb_accuracy = accuracy_score(y_val, cb_val_classes) * 100

print("="*50)
# 4. CatBoost natijalarini chiqaramiz
print(f"🔥 CatBoost modeli Validation LogLoss: {cb_loss:.5f}")
print(f"🎯 CatBoost modeli Accuracy:           {cb_accuracy:.2f}%")
print("="*50)

0:	learn: 1.0393860	test: 1.0397337	best: 1.0397337 (0)	total: 4.82ms	remaining: 2.88s
100:	learn: 0.4056103	test: 0.4156257	best: 0.4156257 (100)	total: 438ms	remaining: 2.17s
200:	learn: 0.3823915	test: 0.3998313	best: 0.3998313 (200)	total: 819ms	remaining: 1.63s
300:	learn: 0.3684217	test: 0.3936113	best: 0.3936113 (300)	total: 1.19s	remaining: 1.18s
400:	learn: 0.3553736	test: 0.3889184	best: 0.3889184 (400)	total: 1.58s	remaining: 783ms
500:	learn: 0.3438311	test: 0.3852989	best: 0.3852989 (500)	total: 1.96s	remaining: 388ms
599:	learn: 0.3343006	test: 0.3829816	best: 0.3829816 (599)	total: 2.34s	remaining: 0us

bestTest = 0.3829815739
bestIteration = 599

🔥 CatBoost modeli Validation LogLoss: 0.38298
🎯 CatBoost modeli Accuracy:           84.80%


In [33]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Datani muvozanatlash (SMOTE) - Kam klasslarni sun'iy ko'paytiramiz
# Faqat o'qitish to'plamini muvozanatlashtiramiz, test to'plamiga tegmaymiz
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("Asl datada klasslar taqsimoti:\n", y.value_counts())
print("SMOTE dan keyingi taqsimot:\n", y_resampled.value_counts())

# 5 talik Stratified K-Fold tayyorlaymiz
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

Asl datada klasslar taqsimoti:
 Status
0    10054
2     4565
1      381
Name: count, dtype: int64
SMOTE dan keyingi taqsimot:
 Status
0    10054
2    10054
1    10054
Name: count, dtype: int64


In [34]:
from sklearn.metrics import classification_report, accuracy_score

# 1. Butun muvozanatlashgan ma'lumotlar to'plamida gibrid modelning klass bashoratlarini olamiz
# O'rganish darajasini aniq tahlil qilish uchun o'rtacha ehtimolliklardan foydalanamiz
xgb_all_preds = xgb.predict_proba(X_resampled)
lgb_all_preds = lgb.predict_proba(X_resampled)
cb_all_preds = cb.predict_proba(X_resampled)

# Gibrid ehtimolliklar
final_all_preds_proba = (xgb_all_preds + lgb_all_preds + cb_all_preds) / 3

# Ehtimolliklardan eng yuqorisini yakuniy klass (0, 1 yoki 2) deb olamiz
final_all_classes = np.argmax(final_all_preds_proba, axis=1)

# 2. Yangi daxshatli ko'rsatkichlarni hisoblaymiz
hybrid_accuracy = accuracy_score(y_resampled, final_all_classes) * 100

print("="*60)
print(f"🔥 YANGI GIBRID MODELNING UMUMIY ANIQLIGI: {hybrid_accuracy:.2f}%")
print("="*60)
print("\n📊 KLASSIFIKATSIYA HISOBOTI (CLASSIFICATION REPORT):")
print(classification_report(y_resampled, final_all_classes, target_names=['Status_C (0)', 'Status_CL (1)', 'Status_D (2)']))
print("="*60)

🔥 YANGI GIBRID MODELNING UMUMIY ANIQLIGI: 90.67%

📊 KLASSIFIKATSIYA HISOBOTI (CLASSIFICATION REPORT):
               precision    recall  f1-score   support

 Status_C (0)       0.89      0.92      0.90     10054
Status_CL (1)       0.93      0.96      0.94     10054
 Status_D (2)       0.90      0.84      0.87     10054

     accuracy                           0.91     30162
    macro avg       0.91      0.91      0.91     30162
 weighted avg       0.91      0.91      0.91     30162



In [35]:

final_submission = pd.DataFrame({
    'id': test_df['id'],
    'Status_C': final_test_preds[:, 0],
    'Status_CL': final_test_preds[:, 1],
    'Status_D': final_test_preds[:, 2]
})

# 3. Faylni 'submission.csv' nomi bilan Kaggle muhitiga saqlaymiz
final_submission.to_csv('submission.csv', index=False)

print("="*50)
print("🏆 YAKUNIY G'OLIBONA FAYL TAYYORLANDI!")
print("="*50)
# Faylning birinchi 5 ta qatorini ko'zimiz bilan tekshirib olamiz
print(final_submission.head())
print("="*50)

🏆 YAKUNIY G'OLIBONA FAYL TAYYORLANDI!
      id  Status_C  Status_CL  Status_D
0  15000  0.106230   0.026212  0.867558
1  15001  0.926335   0.016121  0.057544
2  15002  0.599344   0.020019  0.380638
3  15003  0.964449   0.004805  0.030746
4  15004  0.284130   0.118475  0.597395
